# Conversor Voz - One Click Colab

Execute as celulas em ordem. O notebook primeiro monta o Google Drive e copia os arquivos para `/content/voz_neural`; depois instala as bibliotecas conforme o motor detectado e abre a interface para gerar WAV.


In [ ]:
# 1) Fazer login e montar o Google Drive
from google.colab import drive

drive.mount('/content/drive')
print('Google Drive montado em /content/drive')


In [ ]:
# 2) Criar o modulo do conversor dentro do Colab
import sys
from pathlib import Path
MODULE_CODE = 'from __future__ import annotations\n\nimport os\nimport re\nimport shutil\nimport subprocess\nfrom urllib.request import urlopen\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable\n\n\nDRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/13uBrrPLx--PlHho0fcz5xW8eYehimRcC?usp=sharing"\nCONFIG_FILE_ID = "1y_fKsgq8h_uWVCPDmzc9bnR2vmnJA1Pb"\nTARGET_CHECKPOINT_NAME = "neuralepoch_2nd_00024.pth"\nTARGET_CHECKPOINT_NAMES = (TARGET_CHECKPOINT_NAME, "epoch_2nd_00024.pth")\nMODEL_ROOT = Path("/content/voz_neural")\nOUTPUT_DIR = Path("/content/audios_gerados")\nCOLAB_DRIVE_ROOT = Path("/content/drive")\n\n\n@dataclass(frozen=True)\nclass ModelBundle:\n    engine: str\n    model_path: Path\n    config_path: Path | None = None\n\n\ndef run_command(command: list[str], input_text: str | None = None) -> None:\n    subprocess.run(command, input=input_text, text=True, check=True)\n\n\ndef download_drive_folder(folder_url: str = DRIVE_FOLDER_URL, output_dir: Path = MODEL_ROOT) -> Path:\n    output_dir.mkdir(parents=True, exist_ok=True)\n    run_command(["gdown", "--folder", folder_url, "-O", str(output_dir), "--remaining-ok"])\n    return output_dir\n\n\ndef download_config_file(file_id: str = CONFIG_FILE_ID, output_dir: Path = MODEL_ROOT) -> Path:\n    output_dir.mkdir(parents=True, exist_ok=True)\n    config_path = output_dir / "styletts2_config.yml"\n    url = f"https://drive.google.com/uc?export=download&id={file_id}"\n    try:\n        with urlopen(url, timeout=60) as response:\n            config_path.write_bytes(response.read())\n    except Exception:\n        run_command(["gdown", "--id", file_id, "-O", str(config_path)])\n    return config_path\n\n\ndef infer_model_project_root(checkpoint_path: Path) -> Path:\n    for parent in checkpoint_path.parents:\n        has_styletts_assets = (parent / "Utils").exists() or (parent / "Models").exists()\n        if has_styletts_assets:\n            return parent\n    return checkpoint_path.parent\n\n\ndef copy_tree_contents(source_dir: Path, output_dir: Path) -> None:\n    output_dir.mkdir(parents=True, exist_ok=True)\n    for source_path in source_dir.rglob("*"):\n        relative_path = source_path.relative_to(source_dir)\n        target_path = output_dir / relative_path\n        if source_path.is_dir():\n            target_path.mkdir(parents=True, exist_ok=True)\n        else:\n            target_path.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(source_path, target_path)\n\n\ndef copy_from_mounted_drive(\n    drive_root: Path = COLAB_DRIVE_ROOT,\n    output_dir: Path = MODEL_ROOT,\n) -> bool:\n    if not drive_root.exists():\n        return False\n\n    checkpoint_path = select_checkpoint_path(drive_root)\n    if not checkpoint_path:\n        return False\n\n    source_root = infer_model_project_root(checkpoint_path)\n    print(f"Checkpoint encontrado no Google Drive: {checkpoint_path}")\n    print(f"Copiando arquivos de {source_root} para {output_dir}...")\n    copy_tree_contents(source_root, output_dir)\n    return True\n\n\ndef import_model_files(drive_root: Path = COLAB_DRIVE_ROOT, output_dir: Path = MODEL_ROOT) -> Path:\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    copied_from_drive = copy_from_mounted_drive(drive_root=drive_root, output_dir=output_dir)\n    if not copied_from_drive:\n        print("Checkpoint nao encontrado no Drive montado. Baixando pasta publica com gdown...")\n        download_drive_folder(output_dir=output_dir)\n\n    print("Salvando YAML de configuracao dentro do Colab...")\n    download_config_file(output_dir=output_dir)\n    return output_dir\n\n\ndef iter_files(root: Path) -> Iterable[Path]:\n    if not root.exists():\n        return []\n    return sorted(path for path in root.rglob("*") if path.is_file())\n\n\ndef print_file_report(root: Path) -> None:\n    files = list(iter_files(root))\n    if not files:\n        print("Nenhum arquivo encontrado.")\n        return\n\n    print("Arquivos encontrados:")\n    for path in files:\n        size_mb = path.stat().st_size / 1024 / 1024\n        print(f"- {path} ({size_mb:.1f} MB)")\n\n\ndef nearest_config_for_checkpoint(checkpoint: Path, root: Path, patterns: tuple[str, ...]) -> Path | None:\n    candidates: list[Path] = []\n    for pattern in patterns:\n        candidates.extend(root.rglob(pattern))\n    candidates = sorted(set(candidates))\n    if not candidates:\n        return None\n\n    same_folder = [path for path in candidates if path.parent == checkpoint.parent]\n    if same_folder:\n        return same_folder[0]\n\n    parent_folder = [path for path in candidates if checkpoint.parent.is_relative_to(path.parent)]\n    if parent_folder:\n        return sorted(parent_folder, key=lambda path: len(path.parts), reverse=True)[0]\n\n    return candidates[0]\n\n\ndef checkpoint_epoch(path: Path) -> int:\n    match = re.search(r"epoch_2nd_(\\d+)\\.pth$", path.name)\n    if not match:\n        return -1\n    return int(match.group(1))\n\n\ndef select_checkpoint_path(root: Path) -> Path | None:\n    pth_files = sorted(root.rglob("*.pth"))\n    if not pth_files:\n        return None\n\n    for target_name in TARGET_CHECKPOINT_NAMES:\n        matches = [path for path in pth_files if path.name == target_name]\n        if matches:\n            return sorted(matches)[0]\n\n    epoch_files = [path for path in pth_files if checkpoint_epoch(path) >= 0]\n    if epoch_files:\n        return sorted(epoch_files, key=lambda path: (checkpoint_epoch(path), str(path)), reverse=True)[0]\n\n    return pth_files[0]\n\n\ndef find_coqui_bundle(root: Path) -> ModelBundle | None:\n    model_path = select_checkpoint_path(root)\n    if not model_path:\n        return None\n\n    config_path = nearest_config_for_checkpoint(model_path, root, ("config.json",))\n    if not config_path:\n        return None\n\n    return ModelBundle(engine="coqui", model_path=model_path, config_path=config_path)\n\n\ndef find_styletts2_bundle(root: Path) -> ModelBundle | None:\n    model_path = select_checkpoint_path(root)\n    if not model_path:\n        return None\n\n    config_path = nearest_config_for_checkpoint(model_path, root, ("*.yml", "*.yaml"))\n    if not config_path:\n        return None\n\n    config_text = config_path.read_text(encoding="utf-8", errors="ignore")\n    styletts2_markers = ("ASR_config", "PLBERT_dir", "model_params", "preprocess_params")\n    if not all(marker in config_text for marker in styletts2_markers):\n        return None\n\n    return ModelBundle(engine="styletts2", model_path=model_path, config_path=config_path)\n\n\ndef find_piper_bundle(root: Path) -> ModelBundle | None:\n    onnx_files = sorted(root.rglob("*.onnx"))\n    if not onnx_files:\n        return None\n    return ModelBundle(engine="piper", model_path=onnx_files[0])\n\n\ndef detect_model_bundle(root: Path) -> ModelBundle:\n    coqui = find_coqui_bundle(root)\n    if coqui:\n        return coqui\n\n    styletts2 = find_styletts2_bundle(root)\n    if styletts2:\n        return styletts2\n\n    piper = find_piper_bundle(root)\n    if piper:\n        return piper\n\n    pth_files = sorted(root.rglob("*.pth"))\n    if pth_files:\n        names = ", ".join(path.name for path in pth_files)\n        raise RuntimeError(\n            "Encontrei arquivo .pth, mas nao encontrei uma configuracao suportada. "\n            "Para usar checkpoint .pth como TTS, o Colab precisa da configuracao "\n            f"da arquitetura do treinamento. Arquivos .pth encontrados: {names}"\n        )\n\n    raise RuntimeError(\n        "Nao encontrei modelo suportado. Esperado: StyleTTS2 com .yml/.yaml + .pth, "\n        "Coqui TTS com config.json + .pth, ou Piper com .onnx."\n    )\n\n\ndef patch_torch_load_for_styletts2():\n    import torch\n\n    original_load = torch.load\n\n    def load_with_legacy_checkpoint_support(*args, **kwargs):\n        kwargs.setdefault("weights_only", False)\n        return original_load(*args, **kwargs)\n\n    torch.load = load_with_legacy_checkpoint_support\n    return original_load\n\n\ndef restore_torch_load(original_load) -> None:\n    import torch\n\n    torch.load = original_load\n\n\ndef report_styletts2_auxiliary_paths(config_path: Path) -> None:\n    config_text = config_path.read_text(encoding="utf-8", errors="ignore")\n    relative_paths = re.findall(r"(?:ASR_config|ASR_path|F0_path|PLBERT_dir):\\s*([^,\\n}]+)", config_text)\n    missing_paths = []\n    for relative_path in relative_paths:\n        candidate = config_path.parent / relative_path.strip()\n        if not candidate.exists():\n            missing_paths.append(candidate)\n\n    if missing_paths:\n        print("Aviso: arquivos auxiliares do StyleTTS2 nao encontrados localmente.")\n        print("O pacote styletts2 tentara baixar modelos padrao para esses itens:")\n        for path in missing_paths:\n            print(f"- {path}")\n\n\nclass NeuralVoiceSynthesizer:\n    def __init__(self, bundle: ModelBundle, output_dir: Path = OUTPUT_DIR):\n        self.bundle = bundle\n        self.output_dir = output_dir\n        self.output_dir.mkdir(parents=True, exist_ok=True)\n        self.tts = None\n        self._load()\n\n    def _load(self) -> None:\n        if self.bundle.engine == "coqui":\n            import torch\n            from TTS.api import TTS\n\n            self.tts = TTS(\n                model_path=str(self.bundle.model_path),\n                config_path=str(self.bundle.config_path),\n                progress_bar=True,\n                gpu=torch.cuda.is_available(),\n            )\n            print("Modelo Coqui carregado.")\n            print("Speakers:", getattr(self.tts, "speakers", None))\n            print("Languages:", getattr(self.tts, "languages", None))\n            return\n\n        if self.bundle.engine == "piper":\n            print("Modelo Piper pronto para inferencia.")\n            return\n\n        if self.bundle.engine == "styletts2":\n            report_styletts2_auxiliary_paths(self.bundle.config_path)\n            original_torch_load = patch_torch_load_for_styletts2()\n\n            previous_cwd = Path.cwd()\n            os.chdir(self.bundle.config_path.parent)\n            try:\n                from styletts2 import tts\n\n                self.tts = tts.StyleTTS2(\n                    model_checkpoint_path=str(self.bundle.model_path),\n                    config_path=str(self.bundle.config_path),\n                )\n            finally:\n                os.chdir(previous_cwd)\n                restore_torch_load(original_torch_load)\n            print("Modelo StyleTTS2 carregado.")\n            return\n\n        raise ValueError(f"Engine nao suportada: {self.bundle.engine}")\n\n    @staticmethod\n    def _choose_first(values, preferred: list[str] | None = None):\n        if not values:\n            return None\n        preferred = preferred or []\n        lowered = {str(value).lower(): value for value in values}\n        for item in preferred:\n            if item.lower() in lowered:\n                return lowered[item.lower()]\n        return values[0]\n\n    def _next_wav_path(self) -> Path:\n        index = len(list(self.output_dir.glob("audio_*.wav"))) + 1\n        return self.output_dir / f"audio_{index:04d}.wav"\n\n    def synthesize(self, text: str) -> str:\n        text = (text or "").strip()\n        if not text:\n            raise ValueError("Digite uma frase antes de gerar o audio.")\n\n        wav_path = self._next_wav_path()\n        if self.bundle.engine == "coqui":\n            kwargs = {"text": text, "file_path": str(wav_path)}\n            speakers = getattr(self.tts, "speakers", None)\n            languages = getattr(self.tts, "languages", None)\n            speaker = self._choose_first(speakers)\n            language = self._choose_first(languages, ["pt-br", "pt", "portuguese", "portugues"])\n            if speaker:\n                kwargs["speaker"] = speaker\n            if language:\n                kwargs["language"] = language\n            self.tts.tts_to_file(**kwargs)\n            return str(wav_path)\n\n        if self.bundle.engine == "styletts2":\n            self.tts.inference(text, output_wav_file=str(wav_path), output_sample_rate=24000)\n            return str(wav_path)\n\n        run_command(["piper", "--model", str(self.bundle.model_path), "--output_file", str(wav_path)], input_text=text)\n        return str(wav_path)\n\n\ndef create_gradio_app(synthesizer: NeuralVoiceSynthesizer):\n    import gradio as gr\n\n    def generate(text: str):\n        try:\n            audio_path = synthesizer.synthesize(text)\n            return audio_path, audio_path, f"Audio gerado: {audio_path}"\n        except Exception as exc:\n            return None, None, f"Erro: {exc}"\n\n    with gr.Blocks(title="Voz Neural") as demo:\n        gr.Markdown("# Voz Neural\\nDigite uma frase e pressione Enter para gerar um WAV.")\n        text_box = gr.Textbox(label="Frase", placeholder="Digite sua frase e pressione Enter...", lines=1)\n        audio = gr.Audio(label="Audio gerado", type="filepath")\n        download = gr.File(label="Download do WAV")\n        status = gr.Textbox(label="Status", interactive=False)\n        text_box.submit(generate, inputs=text_box, outputs=[audio, download, status]).then(lambda: "", outputs=text_box)\n    return demo\n\n\ndef main(import_files: bool = False) -> None:\n    root = MODEL_ROOT\n    if import_files:\n        root = import_model_files()\n\n    print_file_report(root)\n\n    bundle = detect_model_bundle(root)\n    print(f"Engine detectada: {bundle.engine}")\n    print(f"Modelo: {bundle.model_path}")\n    if bundle.config_path:\n        print(f"Config: {bundle.config_path}")\n\n    synthesizer = NeuralVoiceSynthesizer(bundle)\n    demo = create_gradio_app(synthesizer)\n    demo.launch(share=True, debug=True)\n\n\nif __name__ == "__main__":\n    main()\n'
Path('/content/conversor_voz_colab.py').write_text(MODULE_CODE, encoding='utf-8')
sys.path.insert(0, '/content')
print('Modulo /content/conversor_voz_colab.py criado.')


In [ ]:
# 3) Importar os arquivos do modelo e salvar dentro do Colab
# Primeiro procura epoch_2nd_00024.pth ou neuralepoch_2nd_00024.pth no Drive montado.
# Se nao encontrar, tenta baixar a pasta publica com gdown na etapa de fallback.
import subprocess
import sys
from conversor_voz_colab import import_model_files, print_file_report, detect_model_bundle

try:
    model_root = import_model_files()
except Exception:
    print('Falha ao importar direto. Instalando gdown para tentar fallback publico...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'gdown'], check=True)
    model_root = import_model_files()

print_file_report(model_root)
bundle = detect_model_bundle(model_root)
print('Engine detectada:', bundle.engine)
print('Modelo:', bundle.model_path)
print('Config:', bundle.config_path)


In [ ]:
# 4) Instalar bibliotecas necessarias depois que os arquivos ja foram importados
import subprocess
import sys

SYSTEM_PACKAGES = ['espeak-ng', 'ffmpeg']
BASE_PACKAGES = ['gradio>=4.44.0', 'pydub>=0.25.1', 'soundfile>=0.12.1', 'pyyaml>=6.0.0']
ENGINE_PACKAGES = {
    'styletts2': ['styletts2'],
    'coqui': ['coqui-tts'],
    'piper': ['piper-tts'],
}

def pip_install(*packages, check=True, capture=False):
    command = [sys.executable, '-m', 'pip', 'install', *packages]
    return subprocess.run(command, check=check, text=True, capture_output=capture)

subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', *SYSTEM_PACKAGES], check=True)
pip_install('-q', '-U', 'pip')

for package in BASE_PACKAGES:
    print('Instalando', package)
    pip_install('-q', '-U', package)

engine_packages = ENGINE_PACKAGES.get(bundle.engine, [])
for package in engine_packages:
    print('Instalando pacote do motor', bundle.engine + ':', package)
    result = pip_install('-U', package, check=False, capture=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f'Falha ao instalar {package}')

if bundle.engine == 'styletts2':
    # Corrige estado binario misto do NumPy, comum quando pip troca pacotes cientificos no Colab.
    print('Reinstalando NumPy compativel para evitar erro numpy.dtype size changed...')
    pip_install('--no-cache-dir', '--force-reinstall', 'numpy==1.26.4')

    for module_name in list(sys.modules):
        if module_name == 'numpy' or module_name.startswith('numpy.'):
            del sys.modules[module_name]

    try:
        import numpy as np
        print('NumPy', np.__version__)
        from styletts2 import tts
        print('styletts2 import ok')
    except Exception as exc:
        raise RuntimeError(
            'Falha ao importar styletts2 depois da instalacao. Se o runtime estiver em Python 3.12, '
            'o pacote styletts2 pode entrar em conflito com dependencias binarias. Reinicie o runtime e execute as celulas novamente; '
            'se persistir, use runtime Python 3.10/3.11 ou instale uma versao de StyleTTS2 compativel com o Colab atual.'
        ) from exc

print('Dependencias instaladas para engine:', bundle.engine)


In [ ]:
# 5) Carregar a voz e abrir a interface
# Digite uma frase na caixa e pressione Enter. O WAV ficara disponivel para ouvir e baixar.
from conversor_voz_colab import main

main(import_files=False)
